Hi! I am a Kaggle beginner and not fluent in English, so this notebook is written in Japanese.

Please use your browser's translation (or DeepL/ChatGPT) to read it!

Hope it helps!

# これまでの進捗（簡易版）

| Ver. | 内容 | CV スコア | LB スコア |
| ---- | ---- | ---- | ---- | ---- |
| 1 | パイプライン構築 | - | 0.57033 |
| 2 | CVの導入（Hold-out検証） | 0.8411 | 0.83389 |

In [ ]:
import keras
import keras_nlp

import pandas as pd

from sklearn.model_selection import train_test_split

import kagglehub

In [ ]:
import kagglehub

path = kagglehub.competition_download("nlp-getting-started")

print("Path to competition files:", path)

In [ ]:
import os

train = pd.read_csv(os.path.join(path, "train.csv"))
test = pd.read_csv(os.path.join(path, "test.csv"))
sample_submission = pd.read_csv(os.path.join(path, "sample_submission.csv"))

# データの概要

In [ ]:
train.head()

In [ ]:
test.head()

In [ ]:
sample_submission.head()

In [ ]:
train.shape

# 実行

In [ ]:
%%time

# 1. データ分割
X_train, X_val, y_train, y_val = train_test_split(
    train["text"].values,
    train["target"].values,
    test_size=0.2,
    random_state=0,
    stratify=train["target"].values,
)

# 2. モデルの準備
preset = "distil_bert_base_en_uncased"
preprocessor = keras_nlp.models.DistilBertPreprocessor.from_preset(
    preset, sequence_length=160
)
model = keras_nlp.models.DistilBertClassifier.from_preset(
    preset, preprocessor=preprocessor, num_classes=2
)

# 3. コンパイル（学習率を少し下げて 1e-5 に設定）
model.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=keras.optimizers.AdamW(learning_rate=1e-5),
    metrics=["accuracy"],
)

# 4. 最も検証精度（val_accuracy）が高かった時点の重みを自動保存する設定
checkpoint_cb = keras.callbacks.ModelCheckpoint(
    filepath="best_model.keras",
    monitor="val_accuracy",
    save_best_only=True,  # ベストスコアの時だけ上書き保存
    mode="max",
)

# 5. 学習（エポック数を 5 に増やしてじっくり回す）
history = model.fit(
    x=X_train,
    y=y_train,
    validation_data=(X_val, y_val),
    batch_size=32,
    epochs=5,
    callbacks=[checkpoint_cb],  # コールバックを指定
)

# 6. 保存された「最も優秀だった状態のモデル」を読み込んで推論
best_model = keras.models.load_model("best_model.keras")
predictions = best_model.predict(test["text"].values)
test["target"] = predictions.argmax(axis=-1)

test[["id", "target"]].to_csv("submission.csv", index=False)
print("ベストモデルによる submission.csv を出力しました！")

In [ ]:
# %%time

# # 1. データの分割（Validation データを作成）
# X_train, X_val, y_train, y_val = train_test_split(
#     train["text"].values,
#     train["target"].values,
#     test_size=0.2,
#     random_state=0,
#     stratify=train["target"].values,
# )

# # 2. モデルと前処理の準備
# preset = "distil_bert_base_en_uncased"
# preprocessor = keras_nlp.models.DistilBertPreprocessor.from_preset(
#     preset, sequence_length=160
# )
# model = keras_nlp.models.DistilBertClassifier.from_preset(
#     preset, preprocessor=preprocessor, num_classes=2
# )

# # 3. コンパイル（純正の keras.optimizers.AdamW を使用）
# # 学習率 2e-5 を指定、デフォルトは大きすぎる
# optimizer = keras.optimizers.AdamW(learning_rate=2e-5)
# loss = keras.losses.SparseCategoricalCrossentropy(from_logits=True)

# model.compile(
#     loss=loss,
#     optimizer=optimizer,
#     metrics=["accuracy"],
# )

# # 4. 学習（ CV 精度の確認）
# history = model.fit(
#     x=X_train,
#     y=y_train,
#     validation_data=(X_val, y_val),
#     batch_size=32,
#     epochs=2,
# )

# # 5. 推論と提出ファイルの作成
# predictions = model.predict(test["text"].values)
# test["target"] = predictions.argmax(axis=-1)

# test[["id", "target"]].to_csv("submission.csv", index=False)
# print("submission.csv を出力しました！")